# 01b — w2v-BERT-2.0 + CTC sur Google Colab (complément du notebook 01)

Le fine-tuning complet de **w2v-BERT-2.0** (≈ 600 M paramètres, meilleur modèle du papier : 21,37 % WER) ne tient pas
sur un GPU de 6 Go : poids, gradients et états de l'optimiseur occupent à eux seuls ≈ 6 à 10 Go. Ce notebook l'entraîne
sur Colab (T4 16 Go suffit) à partir des segments **déjà découpés et contrôlés** en local, puis écrit les prédictions
dev/test au format de `results/preds/`. Le notebook 01 les reprend ensuite dans ses tableaux (section 8).

**Avant la première exécution** (une seule fois), sur Google Drive :

```
MyDrive/
├── tedxtn-tunisian-speech/        # le dossier du projet (code), copié tel quel (quelques Mo)
└── tedxtn/
    └── tedxtn_segments.tar        # archive produite en local (segments + corpus.parquet + vocab CTC + LM KenLM)
```

**Reprise** : une copie du dernier checkpoint est gardée sur Drive (`MyDrive/tedxtn/models/asr_w2v-bert-2.0/resume/`). Si Colab coupe la
session, il suffit de tout relancer : l'archive est ré-extraite dans `/content` (≈ 3 à 5 min) et l'entraînement repart
du dernier checkpoint.

**Ordre** : exécuter toutes les cellules (*Exécution > Tout exécuter*), avec un GPU (*Exécution > Modifier le type
d'exécution > T4 GPU*). Pour un essai rapide (quelques minutes) avant le vrai entraînement : `SMOKE = True`.

Règles du projet respectées ici : le LM KenLM est estimé sur le **train** uniquement ; le checkpoint et les paramètres
α/β du LM sont choisis sur le **dev** ; le test ne sert qu'à l'évaluation finale.

In [ ]:
# --- Interrupteurs -------------------------------------------------------------------------------------------
SMOKE = False          # True : 200 segments de train, 40 steps, pour vérifier que tout tourne (aucun résultat gardé)
RUN_TRAIN = True       # fine-tuning (reprend automatiquement depuis le dernier checkpoint sur Drive)
RUN_DECODE = True      # décodage glouton + grille KenLM sur le dev + prédictions dev/test
MAX_EPOCHS = 10        # le papier : 80 epochs sur HPC ; ici early stopping sur la loss du dev

In [ ]:
import os, subprocess, sys, shutil, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
PROJECT_DRIVE = DRIVE / "tedxtn-tunisian-speech"
ARCHIVE = DRIVE / "tedxtn" / "tedxtn_segments.tar"
assert PROJECT_DRIVE.exists(), f"Copier le dossier du projet dans {PROJECT_DRIVE}"
assert ARCHIVE.exists(), f"Déposer l'archive dans {ARCHIVE}"

# Données lues en local (/content, rapide) ; modèles et résultats écrits sur Drive (persistants).
LOCAL = Path("/content/tedxtn")
OUT_ROOT = DRIVE / "tedxtn" / ("smoke" if SMOKE else "")          # checkpoints (Drive)
# Prédictions et logs directement dans results/ du projet sur Drive : la section 8 du notebook 01 les retrouve
# (results/preds/asr__<système>__<split>.csv). Le smoke test écrit ailleurs pour ne rien mélanger.
RESULTS_OUT = (DRIVE / "tedxtn" / "smoke" / "results") if SMOKE else (PROJECT_DRIVE / "results")
os.environ["TEDXTN_WORKDIR"] = str(LOCAL)
os.environ["TEDXTN_RESULTS_DIR"] = str(RESULTS_OUT)
os.environ["TEDXTN_OPTIM"] = "adamw_bnb_8bit"     # états d'Adam en 8 bits : ≈ 3,6 Go de VRAM économisés
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if SMOKE:
    os.environ.update({"TEDXTN_MAX_STEPS": "40", "TEDXTN_EVAL_STEPS": "20"})

# Code du projet : copie locale (import plus rapide que depuis Drive)
PROJECT = Path("/content/tedxtn-tunisian-speech")
shutil.copytree(PROJECT_DRIVE, PROJECT, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("workdir", ".git", "*.ipynb_checkpoints", "results"))
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "peft==0.21.2",
                "accelerate>=1.0", "bitsandbytes>=0.43", "jiwer>=3.0", "sacrebleu>=2.4", "pyctcdecode>=0.5",
                "https://github.com/kpu/kenlm/archive/master.zip", "soundfile"], check=True)

In [ ]:
# Extraction de l'archive dans /content (à chaque nouvelle session : /content est effacé par Colab)
t0 = time.time()
if not (LOCAL / "data" / "corpus.parquet").exists():
    LOCAL.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xf", str(ARCHIVE), "-C", str(LOCAL)], check=True)
print(f"archive prête en {time.time() - t0:.0f} s :", sorted(p.name for p in LOCAL.iterdir()))
print(subprocess.run(["du", "-sh", str(LOCAL)], capture_output=True, text=True).stdout)

In [ ]:
import json, random, warnings
import numpy as np, pandas as pd, torch
from IPython.display import display

from common import asr as A, data, metrics, text
from common.augment import Augmenter
from common.config import BASE_MODELS, LM_DIR, MODELS_DIR, RESULTS_DIR, SEED, TrainConfig, gpu_memory_gb

warnings.filterwarnings("ignore", category=FutureWarning)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
VRAM = gpu_memory_gb()
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN", f"({VRAM:.1f} Go)")
assert torch.cuda.is_available(), "Choisir un GPU : Exécution > Modifier le type d'exécution"

# Segments : les chemins de corpus.parquet sont ceux de la machine locale -> on les remplace par /content
df = pd.read_parquet(LOCAL / "data" / "corpus.parquet")
df["path"] = [str(LOCAL / "audio_segments" / Path(p).name) for p in df.path]
assert all(Path(p).exists() for p in df.path.sample(50, random_state=0)), "segments manquants dans l'archive"
cfg = TrainConfig()
train_df, rejected = data.clean_for_training(df[df.split == "train"], cfg.min_audio_seconds, cfg.max_audio_seconds)
dev_df = df[df.split == "dev"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)
if SMOKE:   # essai : train réduit ; pseudo-dev/test = segments de train (le dev et le test officiels ne sont pas lus)
    _talks = sorted(train_df.talk.unique())
    dev_df = train_df[train_df.talk == _talks[-1]].head(40).assign(split="dev").reset_index(drop=True)
    test_df = train_df[train_df.talk == _talks[-2]].head(40).assign(split="test").reset_index(drop=True)
    train_df = train_df[train_df.talk.isin(_talks[:-2])].sample(200, random_state=SEED).reset_index(drop=True)
print({s: len(x) for s, x in [("train", train_df), ("dev", dev_df), ("test", test_df)]},
      f"| heures de train : {train_df.duration.sum() / 3600:.2f} h | rejetés : {len(rejected)}")

## Fine-tuning w2v-BERT-2.0 + CTC

Même recette que la section 7 du notebook 01 : vocabulaire de caractères appris sur le train (fourni dans l'archive),
adaptateur de sous-échantillonnage, SpecAugment, augmentation audio (vitesse, bruit, gain), fp16, gradient
checkpointing, optimiseur AdamW 8 bits, early stopping sur la loss du dev, courbe de WER sur 200 segments du dev.
Le batch effectif (32) est conservé quelle que soit la carte : batch par GPU plus petit, accumulation plus grande.

In [ ]:
# --- Espace Drive : suppression DÉFINITIVE ---------------------------------------------------------------------
# Un fichier supprimé depuis le montage /content/drive va dans la corbeille Drive, qui compte dans le quota : chaque
# rotation de checkpoint (≈ 3,5 Go) s'y accumulerait. Les anciennes copies sont donc supprimées par l'API Drive
# (suppression définitive), ce qui demande une autorisation une fois par session (fenêtre Google).
from google.colab import auth
from googleapiclient.discovery import build
auth.authenticate_user()
_DRV = build("drive", "v3", cache_discovery=False)

def _drive_id(path):
    pid = "root"
    for name in Path(path).relative_to(DRIVE).parts:
        q = f"name = '{name}' and '{pid}' in parents and trashed = false"
        r = _DRV.files().list(q=q, fields="files(id)", pageSize=5, spaces="drive").execute().get("files", [])
        if not r:
            return None
        pid = r[0]["id"]
    return pid

def drive_purge(path):
    """Supprime définitivement un fichier ou un dossier de Drive (sans passer par la corbeille)."""
    fid = _drive_id(path)
    if fid:
        _DRV.files().delete(fileId=fid).execute()
    shutil.rmtree(path, ignore_errors=True)   # met à jour la vue locale du montage

def drive_usage_gb():
    q = _DRV.about().get(fields="storageQuota").execute()["storageQuota"]
    return {k: round(int(v) / 1e9, 2) for k, v in q.items() if v is not None}
print("Quota Drive (Go) :", drive_usage_gb())

In [ ]:
# Emplacements (empreinte Drive maximale ≈ archive 2,9 Go + 1 copie de reprise 3,5 Go + meilleur modèle fp16 1,2 Go) :
#   /content/ckpt_w2vbert/                checkpoints du Trainer (disque local de Colab, effacé en fin de session)
#   Drive .../asr_w2v-bert-2.0/resume/    UNE copie du dernier checkpoint (poids fp32 + états Adam 8 bits) : reprise
#   Drive .../asr_w2v-bert-2.0/best_fp16/ meilleur modèle (loss du dev) en fp16, mis à jour à chaque amélioration
#   Drive .../asr_w2v-bert-2.0/final/     modèle final (= best_fp16 renommé) ; resume/ est alors supprimé
CTC_DIR = MODELS_DIR / "asr_w2v-bert-2.0"               # /content : vocabulaire
DRIVE_DIR = OUT_ROOT / "models" / "asr_w2v-bert-2.0"
RESUME_DIR, BEST_DIR, FINAL_DIR = DRIVE_DIR / "resume", DRIVE_DIR / "best_fp16", DRIVE_DIR / "final"
LOCAL_CKPT = Path("/content/ckpt_w2vbert")
MIRROR_EVERY = 1                                          # copie de reprise tous les N checkpoints (1 = tous les 250 steps)
for d in (CTC_DIR, DRIVE_DIR, LOCAL_CKPT):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy(LOCAL / "models" / "asr_w2v-bert-2.0" / "vocab.json", CTC_DIR / "vocab.json")

per_gpu = 4 if VRAM < 20 else 8                         # T4/L4 : 4 ; A100 : 8
os.environ["TEDXTN_MAX_BATCH"] = str(per_gpu)
c = TrainConfig(batch_size=8, grad_accum=4, learning_rate=5e-5, max_epochs=MAX_EPOCHS)
eval_wavs = lambda frame: [data.load_audio(p) for p in frame.path]
sub = dev_df.sample(min(200, len(dev_df)), random_state=SEED)

def save_fp16(model, proc, out):
    """Sauvegarde en fp16 (1,2 Go au lieu de 2,3) ; load_ctc le recharge en fp32 pour l'inférence."""
    out.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out, state_dict={k: v.detach().to("cpu", torch.float16) for k, v in model.state_dict().items()})
    proc.save_pretrained(out)

from transformers import TrainerCallback
class DriveSync(TrainerCallback):
    """Meilleur modèle fp16 et copie de reprise sur Drive, sans jamais garder deux copies à la fois."""
    def __init__(self, proc):
        self.proc, self.n = proc, 0
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kw):
        loss = (metrics or {}).get("eval_loss")
        meta = BEST_DIR / "best.json"
        best = json.loads(meta.read_text())["eval_loss"] if meta.exists() else float("inf")
        if loss is not None and loss < best:
            drive_purge(BEST_DIR)                       # l'ancien meilleur d'abord : pas de double copie sur Drive
            save_fp16(model, self.proc, BEST_DIR)
            meta.write_text(json.dumps({"eval_loss": loss, "step": state.global_step}))
            print(f"[step {state.global_step}] nouveau meilleur modèle (eval_loss {loss:.4f}) -> {BEST_DIR}")
    def on_save(self, args, state, control, **kw):
        self.n += 1
        src = LOCAL_CKPT / f"checkpoint-{state.global_step}"
        if self.n % MIRROR_EVERY or not src.exists():
            return
        for old in RESUME_DIR.glob("checkpoint-*"):
            drive_purge(old)                            # l'ancienne copie d'abord : une seule copie de reprise
        shutil.copytree(src, RESUME_DIR / src.name)

if RUN_TRAIN and (FINAL_DIR / "config.json").exists():
    print("modèle final déjà présent sur Drive : entraînement sauté ->", FINAL_DIR)
elif RUN_TRAIN:
    # reprise : checkpoint local de la session en cours, sinon la copie de reprise sur Drive
    local_last = max(LOCAL_CKPT.glob("checkpoint-*"), default=None, key=lambda p: int(p.name.split("-")[1]))
    drive_last = max(RESUME_DIR.glob("checkpoint-*"), default=None, key=lambda p: int(p.name.split("-")[1]))
    if local_last is None and drive_last is not None:
        local_last = LOCAL_CKPT / drive_last.name
        shutil.copytree(drive_last, local_last)
    print("reprise depuis :", local_last)
    proc, model = A.load_ctc(BASE_MODELS["w2v-bert-2.0"], CTC_DIR, c)
    tr = A.SpeechDataset(train_df, augmenter=Augmenter(seed=SEED))
    dv = A.SpeechDataset(dev_df)
    cb = A.GenerationWERCallback(lambda m, w: A.ctc_transcribe(m, proc, w), eval_wavs(sub), sub.transcription.tolist())
    # save_total_limit=1 avec load_best_model_at_end : le Trainer garde le meilleur ET le dernier (disque local)
    args = A.training_args(LOCAL_CKPT, c, seq2seq=False, save_total_limit=1)
    print(f"batch {args.per_device_train_batch_size} x accumulation {args.gradient_accumulation_steps} | "
          f"optimiseur {args.optim} | eval/save tous les {args.eval_steps} steps")
    trainer = A.make_trainer(model, args, tr, dv, A.CTCCollator(proc), proc, c, [cb.as_callback(), DriveSync(proc)])
    t0 = time.time()
    trainer.train(resume_from_checkpoint=str(local_last) if local_last else None)
    dt = time.time() - t0
    print(f"entraînement : {dt / 3600:.2f} h dans cette session")
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    log_f, cur_f = RESULTS_DIR / "train_log_asr_w2v-bert-2.0.csv", RESULTS_DIR / "dev_wer_curve_asr_w2v-bert-2.0.csv"
    pd.DataFrame(trainer.state.log_history).to_csv(log_f, index=False)   # log_history contient l'historique repris
    prev = pd.read_csv(cur_f) if cur_f.exists() else pd.DataFrame()
    pd.concat([prev, pd.DataFrame(cb.history)]).drop_duplicates("step", keep="last").to_csv(cur_f, index=False)
    # modèle final = meilleur modèle fp16 (renommage, sans copie) ; puis suppression de la copie de reprise
    if not (BEST_DIR / "config.json").exists():
        save_fp16(trainer.model, proc, BEST_DIR)
    if FINAL_DIR.exists():
        drive_purge(FINAL_DIR)
    shutil.move(str(BEST_DIR), str(FINAL_DIR))
    for old in RESUME_DIR.glob("checkpoint-*"):
        drive_purge(old)
    print("modèle final (meilleure loss du dev) :", json.loads((FINAL_DIR / "best.json").read_text()), "->", FINAL_DIR)
    print("Quota Drive (Go) :", drive_usage_gb())
    del model, trainer; torch.cuda.empty_cache()

## Décodage : glouton (protocole du papier) puis beam search + KenLM

Les poids α (LM) et β (bonus de mot) sont choisis sur 300 segments du **dev** ; le meilleur couple est appliqué au dev
complet et au test. Le LM 4-gram (`lm/tedxtn_4gram.bin`) a été estimé en local sur les transcriptions du train.

In [ ]:
if RUN_DECODE:
    proc, model = A.load_ctc(str(FINAL_DIR), None, from_checkpoint=True)
    dev_w, test_w = eval_wavs(dev_df), eval_wavs(test_df)
    for split, frame, w in [("dev", dev_df, dev_w), ("test", test_df, test_w)]:
        hyps = A.ctc_transcribe(model, proc, w)
        data.save_preds("asr", "w2v-bert-2.0-ft", split, frame.seg_id, hyps, frame.transcription)
        if split == "dev":   # le test n'est pas affiché ici : il sera évalué avec tous les systèmes (notebook 01, §8)
            print("dev glouton :", {k: round(v, 2) for k, v in metrics.wer_cer(frame.transcription.tolist(), hyps).items()
                                    if k in ("WER", "CER")})
    lm_bin = LOCAL / "lm" / "tedxtn_4gram.bin"
    sub_idx = np.random.default_rng(SEED).choice(len(dev_df), min(300, len(dev_df)), replace=False)
    grid = []
    for alpha in [0.3, 0.5, 0.8, 1.0]:
        for beta in [0.0, 1.0, 2.0]:
            dec = A.build_kenlm_decoder(proc, lm_bin, alpha, beta)
            h = A.ctc_transcribe(model, proc, [dev_w[i] for i in sub_idx], decoder=dec)
            grid.append({"alpha": alpha, "beta": beta,
                         "WER_dev": metrics.wer_cer(dev_df.transcription.iloc[sub_idx].tolist(), h)["WER"]})
    grid = pd.DataFrame(grid).sort_values("WER_dev"); display(grid.head())
    grid.to_csv(RESULTS_DIR / "01_kenlm_grid.csv", index=False)
    best = grid.iloc[0]
    dec = A.build_kenlm_decoder(proc, lm_bin, best.alpha, best.beta)
    for split, frame, w in [("dev", dev_df, dev_w), ("test", test_df, test_w)]:
        hyps = A.ctc_transcribe(model, proc, w, decoder=dec)
        data.save_preds("asr", "w2v-bert-2.0-ft+kenlm", split, frame.seg_id, hyps, frame.transcription)
        if split == "dev":
            print(f"dev KenLM (α={best.alpha}, β={best.beta}) :",
                  {k: round(v, 2) for k, v in metrics.wer_cer(frame.transcription.tolist(), hyps).items() if k in ("WER", "CER")})
    del model; torch.cuda.empty_cache()

## À rapporter sur la machine locale

- `MyDrive/tedxtn-tunisian-speech/results/` : `preds/asr__w2v-bert-2.0-ft__{dev,test}.csv`,
  `preds/asr__w2v-bert-2.0-ft+kenlm__{dev,test}.csv`, `train_log_asr_w2v-bert-2.0.csv`,
  `dev_wer_curve_asr_w2v-bert-2.0.csv`, `01_kenlm_grid.csv` → à copier dans `results/` du projet local ;
- `MyDrive/tedxtn/models/asr_w2v-bert-2.0/final/` (≈ 1,2 Go, fp16) → à copier dans `workdir/models/asr_w2v-bert-2.0/final/`
  si w2v-BERT est le meilleur système sur le dev (il sera alors exporté en `asr_best` pour l'interface).

Ces fichiers ne doivent pas être publiés (licence CC BY-NC-ND) : Drive privé uniquement.

In [ ]:
assert RESULTS_DIR == RESULTS_OUT, (RESULTS_DIR, RESULTS_OUT)
print("Résultats :", RESULTS_DIR, sorted(p.name for p in (RESULTS_DIR / "preds").glob("asr__w2v*")))
print("Modèle final :", (FINAL_DIR / "config.json").exists(), FINAL_DIR, "| quota Drive (Go) :", drive_usage_gb())
if SMOKE:
    # l'essai écrit des fichiers de taille réelle (copie de reprise ≈ 3,5 Go) : suppression définitive (pas de corbeille)
    drive_purge(OUT_ROOT)
    print("SMOKE terminé : dossier d'essai supprimé définitivement ->", OUT_ROOT, "| quota Drive (Go) :", drive_usage_gb())
    print("Pour le vrai entraînement : SMOKE = False dans la première cellule, puis Exécution > Tout exécuter.")